# Tome C3, Chapter 3: Turing Machines and Computability (Python companion)

Exercises first, solutions at the end. We build Turing machines as Python dictionaries and run them: the palindrome machine of Proposition 3.2 and its quadratic running time, a two-tape machine that does the same job in linear time, descriptions of machines as strings and a universal simulator that runs them, an exhaustive search that recovers the busy beaver values $S(2)=6$, $S(3)=21$ and $S(4)=107$, the explosive Ackermann function, the Cantor pairing, a program that prints itself, and Hennie's crossing sequences, the reason why one tape costs a quadratic factor.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, time, io, contextlib
import matplotlib.pyplot as plt

def words(alphabet, n):
    for k in range(n + 1):
        for w in itertools.product(alphabet, repeat=k):
            yield ''.join(w)

## Exercises

**P3.1 (★) A Turing machine simulator.** Represent a one-tape machine by a dictionary `delta[state, symbol] = (new state, written symbol, move)`. Write a simulator that returns the verdict and the number of steps, implement the palindrome machine of Proposition 3.2, check it on all words of length at most $12$, and plot its worst running time against $n$. Compare with $n^2/2$.

**P3.2 (★★) Two tapes.** Write a simulator for machines with $k$ tapes (Theorem 3.3) and a two-tape palindrome machine that copies its input, then compares it with the copy read backwards. Measure its running time and compare with P3.1.

**P3.3 (★★) Descriptions and a universal machine.** Encode a one-tape machine as a string over $\{0,1\}$, as in Section 3.3, write the decoder, and a function `U(description, w)` that runs the described machine on $w$. Check that the encoding of the palindrome machine, decoded and run, agrees with P3.1, and see what happens when a single $0$ is inserted into or deleted from the description.

**P3.4 (★★) Busy beavers.** Enumerate the two-symbol machines with $n$ states by *tree normal form*: run a machine whose table is still partly empty, and only when it reaches an empty entry, branch on all the ways of filling it, halting included. Recover $S(1)=1$, $S(2)=6$, $S(3)=21$ and $S(4)=107$ (Section 3.8), with a step limit. What can and cannot be concluded from such a search?

**P3.5 (★) Ackermann.** Compute $A(m,n)$ for small $m,n$ without Python's recursion limit getting in the way, check the formulas of Exercise 3.6, count the number of elementary steps, and write the number of decimal digits of $A(4,2)$.

**P3.6 (★) Pairing.** Implement the Cantor pairing $\pi$ of Exercise 3.7 and its inverse with integer square roots only, and check that it is a bijection on a large square. Use it to list all pairs $(M,w)$ of a dovetailing schedule (Proposition 3.10).

**P3.7 (★) A program that prints itself.** Run the two-line Python program of Exercise 3.8 and check that its output is its own text. Then write a self-reproducing program that also prints the number of characters of its own text.

**P3.8 (★★★) Crossing sequences.** Record, for each run of the palindrome machine and each boundary between two cells, the sequence of states in which the head crosses it. Check on the inputs $u\,a^n\,u^R$ with $|u|=n$ the key fact of Hennie's argument (Exercise 3.10): for different $u$ the crossing sequences at a boundary inside the middle block are different. Compare the total length of crossing sequences in the middle block with $n^2$.

## Solutions

### P3.1

In [ ]:
B = '_'

def run_tm(delta, w, start='s', accept='acc', reject='rej', limit=10**6):
    # One-tape machine; returns (verdict, steps). Missing transitions reject.
    tape, h, q, steps = dict(enumerate(w)), 0, start, 0
    while q not in (accept, reject):
        if steps >= limit:
            return 'limit', steps
        key = (q, tape.get(h, B))
        if key not in delta:
            return 'rej', steps
        q, tape[h], mv = delta[key]
        h += 1 if mv == 'R' else -1
        steps += 1
    return ('acc' if q == accept else 'rej'), steps

# The palindrome machine of Proposition 3.2.
PAL = {('s', B): ('acc', B, 'R'), ('back', B): ('s', B, 'R')}
for c in 'ab':
    o = 'ab'.replace(c, '')
    PAL['s', c] = ('r' + c, B, 'R')          # erase the first letter, remember it
    for d in 'ab':
        PAL['r' + c, d] = ('r' + c, d, 'R')  # run to the right end
    PAL['r' + c, B] = ('t' + c, B, 'L')
    PAL['t' + c, c] = ('back', B, 'L')       # last letter matches: erase it
    PAL['t' + c, o] = ('rej', o, 'L')
    PAL['t' + c, B] = ('acc', B, 'L')        # nothing left: odd palindrome
    PAL['back', c] = ('back', c, 'L')

ok, worst = True, {}
for w in words('ab', 12):
    v, t = run_tm(PAL, w)
    ok &= (v == 'acc') == (w == w[::-1])
    worst[len(w)] = max(worst.get(len(w), 0), t)
print("palindrome machine correct on all words of length <= 12:", ok)
print("worst steps:", worst)
ns = list(worst)
plt.figure(figsize=(6, 3.5))
plt.plot(ns, [worst[n] for n in ns], 'o-', label='worst steps of the machine')
plt.plot(ns, [n * n / 2 for n in ns], '--', label='$n^2/2$')
plt.xlabel('n'); plt.legend(); plt.title('One tape: quadratic time'); plt.show()

The worst case is a palindrome, which the machine must erase completely: each round trip crosses the remaining word twice, so the running time is about $2(n+(n-2)+(n-4)+\cdots)\approx n^2/2$, and the measured values follow $n^2/2$ closely. (The numbers are exact: for instance $28$ steps on `abbbba`, the run drawn in the chapter's space-time diagram.)

In [ ]:
print(run_tm(PAL, 'abbbba'))

### P3.2

In [ ]:
def run_ktm(delta, w, k, start='s', accept='acc', reject='rej', limit=10**6):
    # k tapes, the input on tape 1; moves are 'L', 'R' or 'S' (stay).
    tapes = [dict(enumerate(w))] + [{} for _ in range(k - 1)]
    heads, q, steps = [0] * k, start, 0
    while q not in (accept, reject) and steps < limit:
        key = (q, tuple(t.get(h, B) for t, h in zip(tapes, heads)))
        if key not in delta:
            return 'rej', steps
        q, writes, moves = delta[key]
        for i in range(k):
            tapes[i][heads[i]] = writes[i]
            heads[i] += {'L': -1, 'R': 1, 'S': 0}[moves[i]]
        steps += 1
    return ('acc' if q == accept else 'rej'), steps

PAL2 = {}
for c in 'ab':
    PAL2['copy', (c, B)] = ('copy', (c, c), ('R', 'R'))       # copy the input to tape 2
    for d in 'ab':
        PAL2['rew', (c, d)] = ('rew', (c, d), ('L', 'S'))     # rewind tape 1
        PAL2['cmp', (c, d)] = ('cmp' if c == d else 'rej', (c, d), ('R', 'L'))
    PAL2['rew', (c, B)] = ('rew', (c, B), ('L', 'S'))
PAL2['s', (B, B)] = ('acc', (B, B), ('S', 'S'))
for c in 'ab':
    PAL2['s', (c, B)] = ('copy', (c, c), ('R', 'R'))
PAL2['copy', (B, B)] = ('rew', (B, B), ('L', 'L'))
for d in 'ab':
    PAL2['rew', (B, d)] = ('cmp', (B, d), ('R', 'S'))
PAL2['cmp', (B, B)] = ('acc', (B, B), ('S', 'S'))

ok, worst2 = True, {}
for w in words('ab', 12):
    v, t = run_ktm(PAL2, w, 2)
    ok &= (v == 'acc') == (w == w[::-1])
    worst2[len(w)] = max(worst2.get(len(w), 0), t)
print("two-tape machine correct:", ok)
print("worst steps:", worst2)
print("one tape / two tapes at n = 12:", worst[12], "/", worst2[12])

Three passes over the input: copy, rewind, compare, so about $3n$ steps against $n^2/2$ for one tape. Theorem 3.3 says that one tape can simulate the two in time $O(t^2)$; Hennie's theorem (Exercise 3.10, and P3.8) says that for palindromes this square is unavoidable. The extra tapes add no power, only speed, and the speed-up is at most quadratic.

### P3.3

In [ ]:
def encode(delta, symbols=(B, 'a', 'b')):
    # States numbered from 1 (start), 2 = accept, 3 = reject; symbols from 1; moves L = 1, R = 2.
    # Each transition is 0^q 1 0^s 1 0^q' 1 0^s' 1 0^m, transitions separated by 11.
    names = ['s', 'acc', 'rej'] + sorted({q for q, _ in delta} - {'s'})
    num = {q: i + 1 for i, q in enumerate(names)}
    sym = {c: i + 1 for i, c in enumerate(symbols)}
    parts = []
    for (q, c), (q2, c2, m) in sorted(delta.items()):
        parts.append('1'.join('0' * x for x in (num[q], sym[c], num[q2], sym[c2], 1 if m == 'L' else 2)))
    return '11'.join(parts)

def decode(desc, symbols=(B, 'a', 'b')):
    delta = {}
    for part in desc.split('11'):
        blocks = part.split('1')
        if len(blocks) != 5 or any(not b or set(b) != {'0'} for b in blocks):
            return None                      # not a description: by convention, no machine
        q, c, q2, c2, m = (len(b) for b in blocks)
        if max(c, c2) > len(symbols) or m > 2 or (q, c) in delta:
            return None
        name = lambda i: {1: 's', 2: 'acc', 3: 'rej'}.get(i, 'q%d' % i)
        delta[name(q), symbols[c - 1]] = (name(q2), symbols[c2 - 1], 'LR'[m - 1])
    return delta

def U(desc, w, limit=10**6):
    delta = decode(desc)
    return 'rej' if delta is None else run_tm(delta, w, limit=limit)[0]

d = encode(PAL)
print("description of the palindrome machine:", len(d), "bits; it starts", d[:40], "...")
print("U agrees with the machine on all words of length <= 10:",
      all(U(d, w) == run_tm(PAL, w)[0] for w in words('ab', 10)))
same = broken = changed = 0
for i in range(len(d) + 1):
    for d2 in (d[:i] + '0' + d[i:], d[:i] + d[i + 1:] if d[i:i + 1] == '0' else None):
        if d2 is None:
            continue
        if decode(d2) is None:
            broken += 1
        elif all(U(d2, w, 10**4) == U(d, w) for w in words('ab', 7)):
            same += 1
        else:
            changed += 1
print(f"insert or delete one 0: {broken} are not descriptions, {changed} change the language "
      f"on words of length <= 7, {same} do not")

The description is a few hundred bits, and the universal function `U` (decode, then simulate) runs it correctly: this is Theorem 3.5 in Python, the interpreter being the universal machine. Flipping a single bit always breaks the format (it changes the number of blocks), which is why we insert or delete a $0$, changing one number in one transition. Such a mutation often yields a machine, and the counts show that most of these change the language while some do not, for instance those that change the move made when entering the accepting or rejecting state; whether a given mutation changes the language *at all* is a semantic question, undecidable in general by Rice's theorem (Theorem 3.13).

### P3.4

In [ ]:
def busy_beaver(n, limit):
    # Two symbols 0, 1; states 0..n-1; tree normal form. Returns (max steps of a halting machine,
    # number of halting machines, number of machines still running at the limit).
    best, halting, holdouts = 0, 0, 0
    stack = [{}]
    while stack:
        T = stack.pop()
        tape, h, q, steps = {}, 0, 0, 0
        while steps < limit:
            key = (q, tape.get(h, 0))
            if key not in T:
                break
            q, tape[h], mv = T[key]
            h += mv
            steps += 1
        else:
            holdouts += 1
            continue
        # An empty entry is reached: either it is the halting transition (one more step) ...
        halting += 1
        best = max(best, steps + 1)
        if len(T) == 2 * n - 1:
            continue
        # ... or we fill it, introducing new states in order and moving right first (symmetry).
        used = max([0] + [t[0] for t in T.values()])
        for nq in range(min(used + 2, n)):
            for w in (0, 1):
                for mv in ((1,) if not T else (-1, 1)):
                    T2 = dict(T)
                    T2[key] = (nq, w, mv)
                    stack.append(T2)
    return best, halting, holdouts

for n, limit in [(1, 10), (2, 50), (3, 200), (4, 108)]:
    t0 = time.time()
    best, halting, holdouts = busy_beaver(n, limit)
    print(f"n = {n}: longest halting run {best} steps; {halting} halting tables, "
          f"{holdouts} still running after {limit} steps ({time.time() - t0:.1f} s)")

The search finds $1$, $6$, $21$ and $107$. The halting transition counts as a step, as in Radó's convention. For four states the limit $108$ was chosen knowing the answer; with a limit of $500$ the result is the same (and the search takes about a minute).

What the search proves is only a *lower* bound: there are halting machines running $6$, $21$, $107$ steps. To prove $S(n)$ exactly one must show that every holdout never halts, and this is the part no algorithm can do uniformly (Theorem 3.20). For $n=2$ and $3$ the holdouts fall into a few patterns (loops, translated cycles, counters) that can be checked by hand, which Shen Lin did in his 1963 thesis for $n=3$; for $n=4$ it was done by Allen Brady in 1983; for $n=5$, the bbchallenge needed $181\,385\,789$ machines and a dozen kinds of non-halting certificates, checked in Coq.

### P3.5

In [ ]:
def ackermann(m, n):
    # Iterative: the stack holds the pending first arguments. Returns (value, number of steps).
    stack, steps = [m], 0
    while stack:
        m = stack.pop()
        steps += 1
        if m == 0:
            n += 1
        elif n == 0:
            stack.append(m - 1); n = 1
        else:
            stack.append(m - 1); stack.append(m); n -= 1
    return n, steps

for m in range(4):
    print(f"A({m}, n) for n = 0..6:", [ackermann(m, n)[0] for n in range(7)])
print("A(1,n) = n+2:", all(ackermann(1, n)[0] == n + 2 for n in range(50)))
print("A(2,n) = 2n+3:", all(ackermann(2, n)[0] == 2 * n + 3 for n in range(50)))
print("A(3,n) = 2^(n+3)-3:", all(ackermann(3, n)[0] == 2 ** (n + 3) - 3 for n in range(9)))
print("steps to compute A(3, n), n = 0..8:", [ackermann(3, n)[1] for n in range(9)])
print("A(4,0) =", ackermann(4, 0)[0], " A(4,1) = A(3, 13) =", 2 ** 16 - 3, "(by the formula)")
tower = lambda k: 1 if k == 0 else 2 ** tower(k - 1)
print("A(4,n) + 3 = 2^2^...^2 (n+3 twos) for n = 0, 1:", ackermann(4, 0)[0] + 3 == tower(3), 2 ** 16 == tower(4))
a42 = tower(5) - 3   # computed by the formula, 65536 bits
print("A(4,2) = 2^65536 - 3 has", a42.bit_length(), "bits and", math.floor(65536 * math.log10(2)) + 1, "decimal digits")

The rows grow as $n$, $2n$, $2^n$, and then a tower: $A(4,n)=2^{2^{\cdot^{\cdot^{2}}}}-3$ with $n+3$ twos. The number of elementary steps grows even faster than the values: it is multiplied by about $4$ each time $n$ increases by $1$ in $A(3,n)$, so computing $A(4,1)=A(3,13)=65533$ by the recursion would take over a billion steps, which is why we used the formula. $A(4,2)=2^{65536}-3$ has $19\,729$ digits, and $A(4,3)$ has more digits than there are atoms in the observable universe. Theorem 3.17 says that no primitive recursive function, no program made of bounded loops only, grows that fast.

### P3.6

In [ ]:
def pair(x, y):
    return (x + y) * (x + y + 1) // 2 + y

def unpair(z):
    s = (math.isqrt(8 * z + 1) - 1) // 2     # the diagonal x + y = s containing z
    y = z - s * (s + 1) // 2
    return s - y, y

N = 300
codes = {pair(x, y) for x in range(N) for y in range(N)}
print("injective on the square:", len(codes) == N * N)
print("onto the triangle 0..N(N+1)/2-1 and inverted by unpair:",
      all(pair(*unpair(z)) == z and max(unpair(z)) < N for z in range(N * (N + 1) // 2)))
print("dovetailing schedule (machine, number of steps):", [unpair(z) for z in range(15)])

The integer square root finds the diagonal $x+y=s$ containing $z$, because $\frac{s(s+1)}2\le z<\frac{(s+1)(s+2)}2$ is equivalent to $s=\lfloor(\sqrt{8z+1}-1)/2\rfloor$. Read as (machine, number of steps), the sequence $\mathrm{unpair}(0),\mathrm{unpair}(1),\dots$ is a schedule which eventually runs every machine for every number of steps: this is how the enumerator of Proposition 3.10 interleaves infinitely many computations without waiting for any one of them.

### P3.7

In [ ]:
src = "s = 's = %r\\nprint(s %% s)'\nprint(s % s)\n"
print(src)

def output_of(program):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        exec(program, {})
    return buf.getvalue()

print("prints its own text:", output_of(src) == src)

src2 = "s = 's = %r\\nt = s %% s\\nprint(t)\\nprint(len(t) + 1)'\nt = s % s\nprint(t)\nprint(len(t) + 1)\n"
out = output_of(src2)
print(out)
print("first part of the output is the text, then its length:", out == src2 + str(len(src2)) + "\n")

The second program follows the proof of the recursion theorem (Theorem 3.19) to the letter: `t = s % s` is the step "obtain your own description", and what follows is any computation on it, here its length. (The `+ 1` counts the final newline that `print` adds.) The same two lines, with `print(len(t) + 1)` replaced by any code, give a program that can inspect, copy or transform its own text.

### P3.8

In [ ]:
def crossing_sequences(delta, w, start='s', accept='acc', reject='rej'):
    # cs[i] = states entered when the head crosses the boundary between cells i-1 and i.
    tape, h, q = dict(enumerate(w)), 0, start
    cs = {}
    while q not in (accept, reject):
        key = (q, tape.get(h, B))
        q, tape[h], mv = delta.get(key, (reject, tape.get(h, B), 'R'))
        h2 = h + (1 if mv == 'R' else -1)
        cs.setdefault(max(h, h2), []).append(q)
        h = h2
    return cs

for n in range(2, 7):
    ws = [''.join(u) for u in itertools.product('ab', repeat=n)]
    runs = {u: crossing_sequences(PAL, u + 'a' * n + u[::-1]) for u in ws}
    distinct = all(len({tuple(runs[u].get(i, [])) for u in ws}) == len(ws) for i in range(n + 1, 2 * n))
    total = min(sum(len(runs[u].get(i, [])) for i in range(n + 1, 2 * n)) for u in ws)
    print(f"n = {n}: crossing sequences in the middle block distinct for all {len(ws)} words u: {distinct};"
          f" least total length {total} (n^2 = {n * n})")

For every boundary inside the middle block $a^n$, the $2^n$ inputs $u\,a^n\,u^R$ give $2^n$ different crossing sequences. This is the heart of Hennie's proof: if two inputs $u\,a^n\,u^R$ and $v\,a^n\,v^R$ had the same crossing sequence at a boundary, the machine could not tell the left part of one from the left part of the other, and would accept the non-palindrome $u\,a^n\,v^R$ as well. Since $2^n$ different sequences over a finite set of states cannot all be short, for some $u$ the sequences are of length $\Omega(n)$ at a constant fraction of the $n$ boundaries, and the running time, which is at least the total length of all crossing sequences, is $\Omega(n^2)$. For our machine the totals are indeed of order $n^2$; Hennie's theorem says that no cleverness in the design of a one-tape machine can avoid it.